# LDA MODEL WITH SKLEARN
- References:
1. https://medium.com/data-science/practical-guide-to-topic-modeling-with-lda-05cd6b027bdf
2. https://machinelearningplus.com/nlp/topic-modeling-python-sklearn-examples/#1introduction

In [ ]:
import pandas as pd
import spacy
import numpy as np 
from spacy.language import Language
from spacy.schemas import ConfigSchemaNlp
ConfigSchemaNlp.model_rebuild()
nlp = spacy.load('en_core_web_lg')
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import RandomizedSearchCV
from sklearn.datasets import fetch_20newsgroups
from scipy.stats import loguniform, randint, uniform

# Plotting tools
import pyLDAvis
import matplotlib.pyplot as plt
%matplotlib inline
# own function 
import lda_preprocess



# Preprocessing

In [3]:
docs = pd.read_csv('../datasets/final/mda_shared_processed.csv')
dtm, _, vectorizer = lda_preprocess.docs2both(docs)

In [ ]:
## gensim - use dis
_, vectors, vectorizer = lda_preprocess.docs2both('../datasets/final/mda_shared_processed.csv')

# Training Model

In [4]:
lda_model = LatentDirichletAllocation(
    n_components=10,
    learning_method="online",
    total_samples=len(docs),
    batch_size=2000,
    learning_decay=0.7,
    learning_offset=100,
    random_state=42,
)
lda_model.fit(dtm)
lda_output = lda_model.fit_transform(dtm)

## Evaluating Perplexity and Log Likelihood

In [5]:
# Log Likelyhood: Higher the better
print("Log Likelihood: ", lda_model.score(dtm))

# Perplexity: Lower the better. Perplexity = exp(-1. * log-likelihood per word)
print("Perplexity: ", lda_model.perplexity(dtm))

# See model parameters
print(lda_model.get_params())


Log Likelihood:  -229835231.20812532
Perplexity:  4948.606522414649
{'batch_size': 2000, 'doc_topic_prior': None, 'evaluate_every': -1, 'learning_decay': 0.7, 'learning_method': 'online', 'learning_offset': 100, 'max_doc_update_iter': 100, 'max_iter': 10, 'mean_change_tol': 0.001, 'n_components': 10, 'n_jobs': None, 'perp_tol': 0.1, 'random_state': 42, 'topic_word_prior': None, 'total_samples': 18177, 'verbose': 0}


## Random Search for best params

In [ ]:
from scipy.stats import loguniform, randint, uniform

param_dist = {
    "n_components":     randint(5, 40),
    "doc_topic_prior":  loguniform(1e-2, 10),
    "topic_word_prior": loguniform(1e-2, 10),
    "learning_decay":   uniform(0.5, 0.5),
    "max_iter":         randint(10, 50),
}

lda = LatentDirichletAllocation(
    learning_method='online',
    random_state=42,
    n_jobs=1
)

search = RandomizedSearchCV(
    estimator=lda,
    param_distributions=param_dist,
    n_iter=40,
    cv=3,
    random_state=42,
    n_jobs=1,
    refit=True
)

dummy_y = np.zeros(dtm.shape[0])
search.fit(dtm, dummy_y)

print("Best log-likelihood:", search.best_score_)
print("Best params:",         search.best_params_)

### Visualisation to of fine tuning params

In [ ]:
results = pd.DataFrame(search.cv_results_)
results_sorted = results.sort_values("param_n_components")

plt.figure(figsize=(10, 5))
plt.plot(results_sorted["param_n_components"], results_sorted["mean_test_score"], marker='o')
plt.fill_between(
    results_sorted["param_n_components"],
    results_sorted["mean_test_score"] - results_sorted["std_test_score"],
    results_sorted["mean_test_score"] + results_sorted["std_test_score"],
    alpha=0.2
)
plt.axvline(
    x=search.best_params_["n_components"],
    color='red', linestyle='--', label=f'Best n_components = {search.best_params_["n_components"]}'
)
plt.xlabel("Number of Topics (n_components)")
plt.ylabel("Mean Log-Likelihood")
plt.title("LDA Random Search — Log-Likelihood vs Number of Topics")
plt.legend()
plt.tight_layout()
plt.show()

## Best Model


In [ ]:
# Best Model
best_lda_model = model.best_estimator_

# Model Parameters
print("Best Model's Params: ", model.best_params_)

# Log Likelihood Score
print("Best Log Likelihood Score: ", model.best_score_)

# Perplexity
print("Model Perplexity: ", best_lda_model.perplexity(dtm))

# Dominant Topic in each doc

In [ ]:
# Create Document - Topic Matrix
lda_output = best_lda_model.transform(dtm)

# column names
topicnames = ["Topic" + str(i) for i in range(best_lda_model.n_topics)]

# index names
docnames = ["Doc" + str(i) for i in range(docs)]

# Make the pandas dataframe
df_document_topic = pd.DataFrame(np.round(lda_output, 2), columns=topicnames, index=docnames)

# Get dominant topic for each document
dominant_topic = np.argmax(df_document_topic.values, axis=1)
df_document_topic['dominant_topic'] = dominant_topic

# Styling
def color_green(val):
    color = 'green' if val > .1 else 'black'
    return 'color: {col}'.format(col=color)

def make_bold(val):
    weight = 700 if val > .1 else 400
    return 'font-weight: {weight}'.format(weight=weight)

# Apply Style
df_document_topics = df_document_topic.head(15).style.applymap(color_green).applymap(make_bold)
df_document_topics

# Topic's Keywords

In [ ]:
# Topic-Keyword Matrix
df_topic_keywords = pd.DataFrame(best_lda_model.components_)

# Assign Column and Index
df_topic_keywords.columns = vectorizer.get_feature_names()
df_topic_keywords.index = topicnames

# View
df_topic_keywords.head()

### Top words per topic

In [ ]:
# Show top n keywords for each topic
def show_topics(vectorizer=vectorizer, lda_model=lda_model, n_words=20):
    keywords = np.array(vectorizer.get_feature_names())
    topic_keywords = []
    for topic_weights in lda_model.components_:
        top_keyword_locs = (-topic_weights).argsort()[:n_words]
        topic_keywords.append(keywords.take(top_keyword_locs))
    return topic_keywords

topic_keywords = show_topics(vectorizer=vectorizer, lda_model=best_lda_model, n_words=15)     ## change number of topics here   

# Topic - Keywords Dataframe
df_topic_keywords = pd.DataFrame(topic_keywords)
df_topic_keywords.columns = ['Word '+str(i) for i in range(df_topic_keywords.shape[1])]
df_topic_keywords.index = ['Topic '+str(i) for i in range(df_topic_keywords.shape[0])]
df_topic_keywords

# Find topics in new doc

In [ ]:
def predict_topic(text, nlp=nlp):
    # preprocess txt 
    docs = lda_preprocess.preprocess_docs(text)
    
    #vectorise
    dtm, _, vectorizer = lda_preprocess.docs2both(docs)

    # LDA Transform
    topic_probability_scores = best_lda_model.transform(dtm)
    topic = df_topic_keywords.iloc[np.argmax(topic_probability_scores), :].values.tolist()
    return topic, topic_probability_scores

# Predict the topic
mytext = ["insert a 2026 10 report"]
topic, prob_scores = predict_topic(text = mytext)
print(topic)